# build_patent_master

**Objective:** Parse all Lens JSONL exports into a single patent master table with titles, abstracts, family members, citations, and classification codes for downstream NLP feature extraction.

**Inputs:** `../raw_batches/*.jsonl(.gz)`.

**Outputs:** `patent_master.parquet`.

## Imports

In [1]:
# Imports
import json
import gzip
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

## Paths

In [2]:
# Paths
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
JSONL_DIR   = RAW / "lens"
OUTPUT_PATH = PROC / "patent_master.parquet"

## JSONL reader and helpers

In [3]:
# Open a JSONL file (plain or gzipped)
def open_jsonl(path: Path):
    if path.name.endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8-sig")
    return open(path, "rt", encoding="utf-8-sig")

In [4]:
# Return the English-language text from a list of localized items
def get_en_text(items):
    if not items:
        return None
    for it in items:
        if it.get("lang") == "en":
            return it.get("text")
    return None

## Parse a single Lens record into a flat dict

In [5]:
# Parse a single Lens record into a flat dict
def parse_record(rec: dict) -> dict:
    biblio  = rec.get("biblio") or {}
    parties = biblio.get("parties") or {}
    fams    = rec.get("families") or {}
    ext_fam = fams.get("extended_family") or {}
    sim_fam = fams.get("simple_family")   or {}

    fam_members = []
    for m in ext_fam.get("members", []):
        did = m.get("document_id") or {}
        if did.get("jurisdiction") and did.get("doc_number"):
            fam_members.append(f"{did['jurisdiction']}|{did['doc_number']}|{did.get('kind','')}")

    cpc = sorted({
        c.get("symbol")
        for c in (biblio.get("classifications_cpc") or {}).get("classifications", [])
        if c.get("symbol")
    })
    ipc = sorted({
        c.get("symbol")
        for c in (biblio.get("classifications_ipcr") or {}).get("classifications", [])
        if c.get("symbol")
    })

    applicants_raw = parties.get("applicants", []) or []
    applicants = [(a.get("extracted_name") or {}).get("value") for a in applicants_raw]
    applicants = [a for a in applicants if a]
    applicant_countries = sorted({
        a.get("residence") for a in applicants_raw if a.get("residence")
    })

    refs = biblio.get("references_cited") or {}
    bwd_lens, bwd_docids = [], []
    for c in refs.get("citations", []) or []:
        pc = c.get("patcit")
        if pc:
            if pc.get("lens_id"):
                bwd_lens.append(pc["lens_id"])
            did = pc.get("document_id") or {}
            if did.get("jurisdiction") and did.get("doc_number"):
                bwd_docids.append(f"{did['jurisdiction']}|{did['doc_number']}|{did.get('kind','')}")

    prio = (biblio.get("priority_claims") or {}).get("earliest_claim") or {}

    return {
        "lens_id":              rec.get("lens_id"),
        "docdb_id":             rec.get("docdb_id"),
        "jurisdiction":         rec.get("jurisdiction"),
        "doc_number":           rec.get("doc_number"),
        "kind":                 rec.get("kind"),
        "date_published":       rec.get("date_published"),
        "application_date":     (biblio.get("application_reference") or {}).get("date"),
        "priority_date":        prio.get("date"),
        "title_en":             get_en_text(biblio.get("invention_title")),
        "abstract_en":          get_en_text(rec.get("abstract")),
        "applicants":           applicants,
        "applicant_countries":  applicant_countries,
        "cpc_codes":            cpc,
        "ipc_codes":            ipc,
        "family_members":       fam_members,
        "simple_family_size":   sim_fam.get("size"),
        "extended_family_size": ext_fam.get("size"),
        "bwd_cit_lens_ids":     bwd_lens,
        "bwd_cit_docids":       bwd_docids,
        "npl_count":            refs.get("npl_count"),
        "npl_resolved_count":   refs.get("npl_resolved_count"),
        "publication_type":     rec.get("publication_type"),
    }

## Main: parse all files, dedupe, write parquet, run sanity checks

In [6]:
# Main: parse all files, deduplicate, write parquet, run sanity checks
def main():
    files = sorted(p for p in JSONL_DIR.glob("*.jsonl*") if not p.name.startswith("."))
    print(f"Found {len(files)} JSONL files in {JSONL_DIR}")

    records, errors = [], []
    for fpath in files:
        try:
            with open_jsonl(fpath) as f:
                for line_no, line in enumerate(f):
                    if not line.strip():
                        continue
                    try:
                        rec = json.loads(line)
                        records.append(parse_record(rec))
                    except Exception as e:
                        errors.append((fpath.name, line_no, str(e)[:200]))
        except Exception as e:
            errors.append((fpath.name, -1, f"FILE-ERROR: {e}"))

    print(f"Parsed {len(records):,} records  (errors: {len(errors)})")

    df = pd.DataFrame(records)
    n_before = len(df)
    df = df.drop_duplicates(subset="lens_id", keep="first")
    df = df.dropna(subset=["lens_id"]).reset_index(drop=True)
    print(f"Deduplicated: {n_before:,} -> {len(df):,} rows ({len(df.columns)} columns)")

    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    table = pa.Table.from_pandas(df, preserve_index=False)
    pq.write_table(table, OUTPUT_PATH, compression="zstd")
    size_mb = OUTPUT_PATH.stat().st_size / 1e6
    print(f"\nWrote: {OUTPUT_PATH}  ({size_mb:.1f} MB)")

    n = len(df)
    print(f"\nSanity checks (n={n:,}):")
    checks = [
        ("lens_id",                df["lens_id"].notna().sum()),
        ("docdb_id",               df["docdb_id"].notna().sum()),
        ("English abstract",       df["abstract_en"].notna().sum()),
        ("English title",          df["title_en"].notna().sum()),
        ("Application date",       df["application_date"].notna().sum()),
        ("Priority date",          df["priority_date"].notna().sum()),
        (">=1 CPC code",           df["cpc_codes"].apply(lambda x: len(x or []) > 0).sum()),
        (">=1 IPC code",           df["ipc_codes"].apply(lambda x: len(x or []) > 0).sum()),
        (">=1 family member",      df["family_members"].apply(lambda x: len(x or []) > 0).sum()),
        (">=1 backward citation",  df["bwd_cit_lens_ids"].apply(lambda x: len(x or []) > 0).sum()),
    ]
    for label, count in checks:
        pct = 100 * count / n if n else 0
        print(f"  {label:25s}: {count:>9,} / {n:,}  ({pct:5.1f}%)")

    print("\nTop 10 jurisdictions:")
    print(df["jurisdiction"].value_counts().head(10).to_string())

## Entry point

In [ ]:
# Entry point
if __name__ == "__main__":
    main()